# Material-fitted 3D local meshes and nonuniform face partitions

An exact oblique-interface transmission problem verifies physical area weights, tensor diffusion, mixed data and one-sided RT reconstruction. The archived nine patches are not a convergence-rate study.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/f90b05ab84cdf242d5edacdd2e5e51a480c40362a18fbc8700ef0e020ede84cb/63_planar3d-companion.zip"
COMPANION_SHA256 = "f90b05ab84cdf242d5edacdd2e5e51a480c40362a18fbc8700ef0e020ede84cb"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/63_planar3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
from functools import partial
from threadpoolctl import threadpool_limits
from IPython.display import Image, display
from pymhm.meshes.tetrahedron import TetraMesh
from pymhm.fem.traces.triangle_3d import TriangularSkeleton
from pymhm.meshes.fitting import fit_planar_material, planar_face_partitions
from pymhm.recovery.moments_3d import reconstruct_darcy_moments_3d

from examples.planar3d_data import Planar3DData


## Declare local and global forms

The application defines the energy/source and both oriented trace incidences.
Qk Cartesian and Pk tetrahedral coordinates use their shared FEM kernels:

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T, & L_T(v)&=(f,v)_T,\\
b_T(\lambda,v)&=\langle\lambda,v\rangle_{\partial T}, &
c_T(p,\mu)&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The declared `LocalEquations` use C=-B.T, a constant pressure kernel and its
physical integral moment. The additional global `Equation` supplies the weak
Dirichlet pressure functional with its prescribed sign. Neumann data are
outward physical fluxes; a fully Neumann boundary uses an explicit pressure
integral constraint. Generic assembly and field interpretation are separate
calls. The sampled gradient flux remains distinct from the conservative trace.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.tetrahedral_darcy import (
    define_tetrahedral_darcy,
    recover_tetrahedral_darcy,
)

data = Planar3DData()
cube = TetraMesh.unit_cube()
macro = TetraMesh(cube.points, cube.cells[:1])
skeleton = TriangularSkeleton(
    macro, degree=1, face_partitions=planar_face_partitions(macro, data.material)
)
fine = fit_planar_material(macro, data.material).mesh
face = int(macro.boundary_faces[0])
with threadpool_limits(1):
    solution_definition = define_tetrahedral_darcy(
        macro,
        degree=4,
        local_meshes=(fine,),
        skeleton=skeleton,
        permeability=data.material,
        source=data.source,
        dirichlet=data.pressure,
        neumann={face: partial(data.normal_flux, normal=macro.normals[face])},
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_tetrahedral_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    recovered = reconstruct_darcy_moments_3d(solution, degree=1)
    errors = dict(
        pressure=solution.l2_error(data.pressure),
        flux=solution.flux_l2_error(data.flux),
        reconstructed=recovered.flux_l2_error(data.flux),
    )
assert max(errors.values()) < 1e-10
errors


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    directory = ROOT / "examples/results/planar3d"
    report = json.loads((directory / "campaign.json").read_text())
    assert len(report["rows"]) == 9
    for row in report["rows"]:
        assert hashlib.sha256((directory / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
        assert max(row["pressure_l2"], row["raw_flux_l2"], row["reconstructed_flux_l2"]) < 2e-8
    [(r["macro_subdivisions"], r["boundary"], r["pressure_l2"], r["raw_flux_l2"],
      r["reconstructed_flux_l2"]) for r in report["rows"]]

## Actual geometry and physical fields

The section is z=0.37. Black/white lines show original macroface intersections. Flat values retain fine-cell ownership; error color scales describe floating-point errors in these exactly represented fields.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("geometry", "fields", "components"):
        display(Image(filename=str(ROOT / "docs/figures/planar3d" / f"{name}.png")))